# QRL Stability Analysis\n\n## Introduction to Quantum Reinforcement Learning\n\nThis notebook analyzes how a parameterized quantum policy learns a navigation task under ideal, noisy, and noise-mitigated execution.

## Policy Gradient Algorithms

We optimize a stochastic policy using REINFORCE with an action-independent baseline. The implemented estimator uses reward-to-go $G_t$, a moving baseline $b_t$, and entropy regularization:

$$

abla_\theta \mathcal{L}_{\mathrm{pg}} = -\sum_t A_t \, \nabla_\theta \log \pi_\theta(a_t\mid s_t) - \beta \sum_t \nabla_\theta H(\pi_\theta(\cdot\mid s_t)), \qquad A_t = G_t - b_t
$$

The current implementation is measurement-defined: the circuit prepares $|\psi_{\theta,s}\rangle = U_\theta(s)|0\rangle$, a dedicated action register is measured in the computational basis, and the policy is the Born distribution $\pi_\theta(a\mid s)=\Pr[M_{\mathrm{action}}=a]$. Parameter-shift is applied directly to action probabilities rather than to expectation-value logits.


## Parameterized Quantum Circuits

The policy uses hybrid angle embedding (`RY`, `RZ`) followed by a configurable variational ansatz (`RealAmplitudes`/`EfficientSU2`). The measured action register defines the policy directly, while the remaining qubits act as latent workspace for the variational circuit.


## Noise in NISQ Devices\n\nThe noisy setting uses an IBM-style noise model (default: `ibm_osaka`) to emulate gate errors, readout errors, and decoherence.

## Impact of Decoherence on Gradient Descent

Noisy sampling distorts the measured Born probabilities of the action register and introduces variance into the policy-gradient estimate, which can slow or destabilize convergence. The mitigation stack therefore acts on measured action probabilities through readout correction and ZNE-style extrapolation.


## Noise Mitigation with Readout Inversion and ZNE\n\n- Readout mitigation uses the affine inverse for single-qubit `Z` expectations: if `m = (1-p_{01}-p_{10}) z + (p_{01}-p_{10})`, then `z = (m-(p_{01}-p_{10}))/(1-p_{01}-p_{10})`.\n- ZNE evaluates observables at folded circuits and extrapolates against the **realized** noise-amplification factors, not just the requested ones.

In [ ]:
from pathlib import Path\nimport json\nimport matplotlib.pyplot as plt\nfrom PIL import Image\nfrom IPython.display import display\n\nRESULTS = Path('../results').resolve()\nsummary_path = RESULTS / 'summary.json'\nif summary_path.exists():\n    summary = json.loads(summary_path.read_text())\n    summary\nelse:\n    print('Run training first: python src/training_pipeline.py --config config/training_config.yaml')

## Learning Curve Analysis

In [ ]:
for image_name in ['learning_curves.png', 'convergence_comparison.png', 'final_policy_plot.png']:\n    image_path = RESULTS / image_name\n    if image_path.exists():\n        display(Image.open(image_path))\n    else:\n        print(f'Missing: {image_path.name}')

## Final Conclusions\n\nComparing ideal, noisy, and mitigated settings quantifies how hardware noise impacts policy optimization and how mitigation can recover learning stability.